# Theory Questions: Energy-Based Models (EBM)

**Q1.1:** To generate a face of a young man using an EBM trained on faces, we can use conditional sampling or guide the sampling process by conditioning on attributes (e.g., age, gender) if available. If not, we can use attribute classifiers to filter or steer the generated samples.

**Q1.2:** Rejection Sampling is a method to sample from a target distribution $p(x)$ using a proposal distribution $q(x)$ and a constant $M$ such that $p(x) \leq M q(x)$ for all $x$. We sample $x \sim q(x)$ and accept with probability $p(x)/(M q(x))$. If $M=2$, the acceptance rate is $1/2$ (i.e., $1/M$).

**Q1.3:**
a) The training process pushes the energy $E_\theta(x)$ down for real data and up for generated (fake) samples.
b) The expectation over the model (second term) is intractable due to the partition function. Contrastive Divergence (CD) approximates this by running a short Markov chain (e.g., Langevin) starting from data, making training feasible.

# Detailed Theory: Energy-Based Models (EBM)

## Q1.1 — Generating a face with a specific attribute (e.g., a young man)

Short answer: use a conditional EBM or steer the sampling via attribute conditioning (classifier guidance or conditioning variables).

Explanation: If the training set contains attribute labels (age, gender, etc.), you can train an EBM conditioned on the attribute y, i.e., model p_θ(x | y) ∝ exp(-E_θ(x, y)). During sampling (e.g., Langevin dynamics) you fix y to the desired attribute and sample x that minimizes the conditional energy. If labels are not available, you can use a pretrained attribute classifier C(x) and either: (a) run classifier-guided Langevin (adjust gradients to increase classifier score for the desired attribute), or (b) reject samples that don’t match the attribute. Both are practical approaches used in conditional generation and guided sampling.

## Q1.2 — Rejection Sampling (explanation + acceptance rate)

Rejection sampling algorithm (brief):
1. Choose a proposal distribution q(x) and a constant M such that p(x) ≤ M q(x) for all x.
2. Repeat: draw x ∼ q(x), draw u ∼ Uniform(0, 1). Accept x if u ≤ p(x) / (M q(x)).

Acceptance probability: E_q[ p(x) / (M q(x)) ] = (1/M) ∫ p(x) dx = 1/M (for normalized p). So the expected acceptance rate equals 1/M. If M = 2, the acceptance rate is 1/2 (50%).

## Q1.3 — Gradient of the log-likelihood and training behavior

Recall (sketch) that for an unnormalized energy model p_θ(x) = exp(-E_θ(x)) / Z_θ, the gradient of the log-likelihood (for a single data point x) can be written as:

∇_θ log p_θ(x) = -∇_θ E_θ(x) + E_{x'∼p_θ}[∇_θ E_θ(x')].

a) What training tries to do:
- The first term (−∇_θ E_θ(x)) pushes parameters so that the energy of data x is decreased (makes data more likely).
- The second term (the model expectation) pushes parameters so that the energy assigned to typical model-generated samples is increased (reduces probability mass in regions where the model currently places mass incorrectly).

Net effect: energies at data points are lowered while energies at model-generated points are raised — this separates data and model distributions.

b) Practical challenge and Contrastive Divergence (CD):
- The expectation E_{x'∼p_θ}[·] requires sampling from p_θ, which is intractable because p_θ depends on the unknown normalization constant Z_θ and may require long MCMC chains to mix.
- Contrastive Divergence approximates this expectation by initializing a short MCMC chain from a data point (or small number of steps from data) and using the resulting sample as an approximate draw from the model. CD is biased but computationally efficient and often works well in practice for training EBMs and related undirected models.

# CA3 HW3: Energy-Based and Score-Based Models on MNIST

**Objectives**

- Implement EBM with Langevin sampling and contrastive divergence.
- Implement NCSN with weighted DSM and annealed Langevin dynamics (unconditional + conditional).
- Provide training, sampling, and denoising pipelines with reproducibility hooks.

**Structure**

1. Setup and configuration
2. Data loading and visualization
3. EBM model, training, sampling, denoising
4. NCSN model, training, sampling (ALD), denoising, conditional variant
5. Results logging placeholders
6. Reproducibility notes


In [ ]:
# Setup and Configuration
import os, random
import numpy as np
import torch
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "data" / "mnist"


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Enable relative imports in notebook
import sys
sys.path.insert(0, str(Path.cwd()))

In [ ]:
# Install requirements (run this in Colab)
!pip install -r ../requirements.txt

In [ ]:
# Configs
from config import DataConfig, EBMConfig, NCSNConfig, RunPaths

data_cfg = DataConfig()
ebm_cfg = EBMConfig(device=device)
ncsn_cfg = NCSNConfig(device=device)
paths = RunPaths()
paths.ensure()
data_cfg, ebm_cfg, ncsn_cfg

In [ ]:
# Data loading and a quick peek
from data import mnist_dataloaders
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt

train_loader, test_loader = mnist_dataloaders(data_cfg, normalize_to_minus1_1=False)
images, labels = next(iter(train_loader))
grid = make_grid(images[:16], nrow=4)
save_image(grid.detach().cpu(), paths.images / "mnist_sample.png")
plt.figure(figsize=(4, 4))
plt.axis("off")
img_arr = grid.permute(1, 2, 0).detach().cpu().numpy()
plt.imshow(img_arr)
plt.show()

In [ ]:
# EBM model, sampler, and a short training utility (configurable)
# Prefer using script helper `train_interactive` from `ebm_train` for reuse
from ebm_model import ConvEnergyModel
from ebm_sampling import LangevinSampler, sample_from_noise
from torch import optim
from tqdm import tqdm
from torchvision.utils import save_image
from ebm_train import train_interactive as train_ebm

# Note: `train_ebm` now refers to `train_interactive` in `ebm_train.py` which
# performs a short interactive run, saves sample grids to `paths.images` and
# returns (model, history). Use `full_train_ebm` from `ebm_train` for full runs.

In [ ]:
# Full EBM Training Pipeline
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Any
from torch import optim
from tqdm import tqdm
from config import DataConfig, EBMConfig, RunPaths
from data import mnist_dataloaders
from ebm_model import ConvEnergyModel
from ebm_sampling import LangevinSampler, sample_from_noise
from utils import save_grid, set_seed, ensure_dir, write_run_info


def full_train_ebm(cfg_data: DataConfig, cfg_model: EBMConfig, output_dir: Path) -> Dict[str, Any]:
    set_seed(cfg_data.seed)
    train_loader, test_loader = mnist_dataloaders(cfg_data)
    device = cfg_model.device

    model = ConvEnergyModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg_model.lr)
    sampler = LangevinSampler(model, cfg_model)

    history = {"loss": [], "E_real": [], "E_fake": []}
    ensure_dir(output_dir)
    checkpoint_path = output_dir / "ebm_ckpt.pt"
    write_run_info(
        output_dir,
        configs={"data": asdict(cfg_data), "model": asdict(cfg_model)},
        notes={"script": "notebook full_train_ebm"},
        device=str(device),
    )

    for epoch in range(1, cfg_model.epochs + 1):
        progress = tqdm(
            train_loader, desc=f"EBM Epoch {epoch}/{cfg_model.epochs}", leave=False
        )
        for step, (x_real, _) in enumerate(progress, start=1):
            x_real = x_real.to(device)
            x_fake = sampler(torch.rand_like(x_real))

            E_real = model(x_real)
            E_fake = model(x_fake)

            data_term = E_real.mean() - E_fake.detach().mean()
            reg_term = cfg_model.lambda_reg * (
                E_real.pow(2).mean() + E_fake.detach().pow(2).mean()
            )
            loss = data_term + reg_term

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            history["loss"].append(loss.item())
            history["E_real"].append(E_real.mean().item())
            history["E_fake"].append(E_fake.mean().item())

            if step % cfg_model.log_interval == 0:
                progress.set_postfix(loss=f"{loss.item():.3f}")

        # Save training samples each epoch
        # Sampling requires gradients to compute input gradients via autograd
        samples = sample_from_noise(
            model, cfg_model, (cfg_model.sample_grid, 1, 28, 28)
        )
        save_grid(samples.detach().cpu(), output_dir / f"ebm_samples_epoch{epoch}.png", nrow=4)

        # Denoising a few test digits via Langevin starting from noisy images
        x_test, _ = next(iter(test_loader))
        x_test = x_test[: cfg_model.sample_grid].to(device)
        noise = torch.randn_like(x_test) * 0.3
        noisy = (x_test + noise).clamp(0.0, 1.0)
        denoised = sampler(noisy)
        save_grid(x_test.detach().cpu(), output_dir / f"ebm_real_epoch{epoch}.png", nrow=4)
        save_grid(noisy.detach().cpu(), output_dir / f"ebm_noisy_epoch{epoch}.png", nrow=4)
        save_grid(denoised.detach().cpu(), output_dir / f"ebm_denoised_epoch{epoch}.png", nrow=4)

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
            },
            checkpoint_path,
        )

    # Save simple visualizations of loss and energies
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6, 4))
        plt.plot(history["loss"], label="loss")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("EBM Loss")
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / "ebm_loss.png")
        plt.close()

        plt.figure(figsize=(6, 4))
        plt.plot(history["E_real"], label="E_real")
        plt.plot(history["E_fake"], label="E_fake")
        plt.xlabel("Step")
        plt.ylabel("Energy")
        plt.title("EBM Energies")
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / "ebm_energy.png")
        plt.close()
    except Exception:
        # Best-effort visualization; continue even if matplotlib is unavailable.
        pass

    return history


# Example: full_ebm_cfg = EBMConfig(epochs=10)
# full_train_ebm(DataConfig(), full_ebm_cfg, paths.images / "ebm")

In [ ]:
# NCSN model, DSM loss, and ALD sampling utilities
# Prefer using script helper `train_interactive` from `ncsn_train` for reuse
from ncsn_model import ScoreNet
from ncsn_loss import dsm_loss
from ncsn_sampling import sample as ncsn_sample
from torchvision.utils import save_image
from ncsn_train import train_interactive as train_ncsn

# Note: `train_ncsn` now refers to `train_interactive` in `ncsn_train.py` which
# performs a short interactive run, saves sample grids to `paths.images` and
# returns (model, history). Use `full_train_ncsn` from `ncsn_train` for full runs.

In [ ]:
# Full NCSN Training Pipeline
import matplotlib.pyplot as plt
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Any, Optional

import torch
from torch import optim
from tqdm import tqdm

from config import NCSNConfig, DataConfig, RunPaths
from data import mnist_dataloaders
from ncsn_model import ScoreNet
from ncsn_loss import dsm_loss
from ncsn_sampling import sample
from utils import save_grid, set_seed, ensure_dir, write_run_info


def full_train_ncsn(cfg: NCSNConfig, output_dir: Path, conditional: bool = False) -> Dict[str, Any]:
    cfg.conditional = conditional
    set_seed(42)

    data_cfg = DataConfig(
        batch_size=cfg.batch_size, num_workers=cfg.num_workers, channels=cfg.channels
    )
    train_loader, _ = mnist_dataloaders(data_cfg, normalize_to_minus1_1=True)

    device = cfg.device
    model = ScoreNet(cfg).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    sigmas = cfg.sigmas

    ensure_dir(output_dir)
    history = {"loss": []}
    checkpoint_path = output_dir / ("ncsn_cond.pt" if conditional else "ncsn.pt")
    write_run_info(
        output_dir,
        configs={
            "data": asdict(data_cfg),
            "model": asdict(cfg),
            "conditional": {"enabled": conditional},
        },
        notes={"script": "notebook full_train_ncsn"},
        device=str(device),
    )

    for epoch in range(1, cfg.epochs + 1):
        progress = tqdm(
            train_loader, desc=f"NCSN Epoch {epoch}/{cfg.epochs}", leave=False
        )
        for x, labels in progress:
            x = x.to(device)
            x = x * 2 - 1  # map to [-1, 1]
            y = labels.to(device) if conditional else None

            loss = dsm_loss(model, x, cfg, sigmas, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            history["loss"].append(loss.item())
            progress.set_postfix(loss=f"{loss.item():.3f}")

        with torch.no_grad():
            y_samples: Optional[torch.Tensor] = None
            if conditional:
                y_samples = torch.arange(0, 16, device=device) % cfg.num_classes
            samples = sample(model, cfg, num_samples=16, y=y_samples)
            samples = (samples + 1) / 2.0
            save_grid(samples, output_dir / f"samples_epoch{epoch}.png", nrow=4)

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
            },
            checkpoint_path,
        )

    # Save loss visualization
    try:
        plt.figure(figsize=(6, 4))
        plt.plot(history["loss"], label="DSM loss")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("NCSN DSM Loss")
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / "ncsn_loss.png")
        plt.close()
    except Exception:
        # Best-effort plotting
        pass

    return history


# Example: full_ncsn_cfg = NCSNConfig(epochs=30)
# full_train_ncsn(full_ncsn_cfg, paths.images / "ncsn", conditional=False)
# full_train_ncsn(full_ncsn_cfg, paths.images / "ncsn_cond", conditional=True)

## Usage Notes

- Full training pipelines are available in the notebook: `full_train_ebm` and `full_train_ncsn`.
- Inference pipelines: `ebm_generate_and_denoise` and `ncsn_generate_and_denoise`.
- For quick demos, use the short training functions with `epochs=1`.
- Ensure `torch` and `torchvision` are installed (see `requirements.txt`).
- Figures are saved to `paths.images` subdirectories; inline plots are for demos.

# Theory Questions: Score-Based Models (NCSN)

**Q2.1:** The score function is $s_\theta(x) = \nabla_x \log p_\theta(x) = -\nabla_x E_\theta(x)$. It is independent of the partition function $Z$ because $\nabla_x \log Z = 0$. This makes training easier since we do not need to compute $Z$.

**Q2.2:** Calculating $\nabla_x \cdot s_\theta(x)$ (divergence) is expensive in high dimensions. DSM replaces this with a denoising objective, which is equivalent to score matching on the noisy distribution, making it tractable.

**Q2.3:** In each region, the score depends only on the local density, not the mixture weights. Thus, Langevin dynamics cannot sample according to the mixture weights and cannot move between disjoint modes.

**Q2.4:** Direct score matching on real data fails due to low support and non-smoothness. Multi-scale noise perturbation (as in NCSN) smooths the data and enables learning. $\sigma_{max}$ controls the largest noise (most smoothing), $\sigma_{min}$ the finest details in annealed Langevin dynamics.

# Detailed Theory: Score-Based Models (NCSN)

## Q2.1 — Score function and independence from the partition function

Definition: the score function is s(x) = ∇_x log p(x). For an energy parameterization p(x) ∝ exp(-E(x)),

log p(x) = -E(x) - log Z, so ∇_x log p(x) = -∇_x E(x) - ∇_x log Z = -∇_x E(x).

Because Z does not depend on x, ∇_x log Z = 0. Therefore the score depends only on E (or the unnormalized density), not on Z. Advantage: learning the score avoids computing or approximating Z (the partition function), which is usually intractable for high-dimensional models.

## Q2.2 — Why the divergence term is hard, and Denoising Score Matching (DSM)

Original score-matching objectives involve the trace (divergence) term ∇_x·s(x) = ∑_i ∂ s_i / ∂ x_i which requires computing many partial derivatives (second derivatives of the underlying model) and scales poorly with dimensionality (images have very large D).

Denoising Score Matching (DSM) idea (intuitive): instead of directly minimizing an objective containing divergence, we corrupt data with Gaussian noise and minimize the expected squared error between the model score and the true score of the noisy distribution: E_{x, x̃∼N(x,σ^2I)} || s_θ(x̃) − ∇_{x̃} log p_σ(x̃ | x) ||^2. This avoids the divergence term and is tractable because the score of the noisy conditional is known analytically for Gaussian noise. In practice this gives an efficient surrogate for the original score-matching objective.

## Q2.3 — Mixture of Gaussians with disjoint supports (effect of mixing weights)

Consider p(x) = π_A p_A(x) + π_B p_B(x). If supports are disjoint (region A where p_B=0 and region B where p_A=0), then for x ∈ A:

log p(x) = log(π_A p_A(x)) = log π_A + log p_A(x),
so ∇ log p(x) = ∇ log p_A(x) (the constant log π_A disappears).

Therefore the score in region A depends only on the local component density p_A, not the mixing weight π_A. Practical consequence: Langevin dynamics uses local gradients and cannot account for global mixing weights, so it will not sample the relative frequencies of the mixture correctly and modes remain isolated (mixing problem). The sampler rarely jumps between disjoint modes because the gradient signal points inward toward the mode and the energy barrier between modes is high.

## Q2.4 — Problems with score matching on raw data and how NCSN fixes them

Problems with direct score matching on real data:
- Real data often lie on a low-dimensional manifold embedded in high-dimensional space; the density off the manifold is effectively zero and the score can be ill-defined or numerically unstable.
- The empirical distribution is spiky; learning the score at exact data points does not generalize well to neighborhoods.

NCSN / Multi-scale noise solution:
- Perturb the data with multiple noise levels (a geometric sequence of σ values). This smooths the data distribution at different scales and makes the score well-defined everywhere.
- Train a model to predict scores conditioned on the noise level (Noise Conditional Score Network).
- Annealed Langevin Dynamics (ALD) sampling: start from large noise σ_max (smooth, easy to explore) and gradually reduce noise to σ_min, performing Langevin updates at each noise level. σ_max encourages global exploration and helps cross modes; σ_min recovers fine details and makes samples sharp.

## References and further reading
- Hyv arinen, "Estimation of non-normalized statistical models by score matching" (2005).
- Song & Ermon, "Generative Modeling by Estimating Gradients of the Data Distribution" (NCSN, 2019).
- Vincent, "A Connection Between Score Matching and Denoising Autoencoders" (2011).

In [ ]:
## Visualization helpers (save and display rich visualizations)
import imageio
import matplotlib.pyplot as plt
from pathlib import Path


def show_image(path: Path, figsize=(5, 5)):
    img = plt.imread(path)
    plt.figure(figsize=figsize)
    plt.axis('off')
    plt.imshow(img)
    plt.show()


def make_gif_from_frames(frames_dir: Path, out_path: Path, fps: int = 6):
    frames = sorted(frames_dir.glob("*.png"))
    if not frames:
        print(f"No frames found in {frames_dir}")
        return
    imgs = [imageio.imread(str(p)) for p in frames]
    imageio.mimsave(str(out_path), imgs, fps=fps)
    print(f"Saved GIF: {out_path}")


def display_saved_run_images(run_images_dir: Path):
    """Display all images saved in a run directory in sorted order."""
    files = sorted(run_images_dir.glob("*.png"))
    for f in files:
        show_image(f)


# Note: imageio is optional; if not available, the GIF creation will raise an ImportError.

In [ ]:
# EBM Inference Pipeline: Generation and Denoising
from pathlib import Path
import torch

from config import DataConfig, EBMConfig, RunPaths
from data import mnist_dataloaders
from ebm_model import ConvEnergyModel
from ebm_sampling import sample_from_noise, LangevinSampler
from utils import save_grid, set_seed, ensure_dir


def load_ebm_model(checkpoint: Path, device: torch.device) -> ConvEnergyModel:
    model = ConvEnergyModel().to(device)
    state = torch.load(checkpoint, map_location=device)
    model.load_state_dict(state["model"])
    model.eval()
    return model


def ebm_generate_and_denoise(
    checkpoint: Path, output_dir: Path, data_cfg: DataConfig, ebm_cfg: EBMConfig
) -> None:
    set_seed(data_cfg.seed)
    train_loader, _ = mnist_dataloaders(data_cfg)
    ensure_dir(output_dir)
    model = load_ebm_model(checkpoint, ebm_cfg.device)
    sampler = LangevinSampler(model, ebm_cfg)

    # Sampling requires gradients (Langevin uses autograd on inputs); do not disable grads here.
    samples = sample_from_noise(model, ebm_cfg, (16, 1, 28, 28))
    save_grid(samples.detach().cpu(), output_dir / "ebm_samples_final.png", nrow=4)

    # Denoise a few training digits
    x_real, _ = next(iter(train_loader))
    x_real = x_real[:16].to(ebm_cfg.device)
    noise = torch.randn_like(x_real) * 0.3
    noisy = (x_real + noise).clamp(0.0, 1.0)
    denoised = sampler(noisy)
    save_grid(x_real.detach().cpu(), output_dir / "ebm_real.png", nrow=4)
    save_grid(noisy.detach().cpu(), output_dir / "ebm_noisy.png", nrow=4)
    save_grid(denoised.detach().cpu(), output_dir / "ebm_denoised.png", nrow=4)


# Example: ebm_generate_and_denoise(paths.images / "ebm" / "ebm_ckpt.pt", paths.images / "ebm_infer", DataConfig(), EBMConfig())

In [ ]:
# NCSN Inference Pipeline: Sampling and Denoising
from pathlib import Path
from typing import Optional, Sequence
import torch

from config import NCSNConfig, DataConfig, RunPaths
from data import mnist_dataloaders
from ncsn_model import ScoreNet
from ncsn_sampling import sample, annealed_langevin_dynamics
from utils import save_grid, ensure_dir


def load_ncsn_model(
    checkpoint: Path, cfg: NCSNConfig, conditional: bool = False
) -> ScoreNet:
    cfg.conditional = conditional
    model = ScoreNet(cfg).to(cfg.device)
    state = torch.load(checkpoint, map_location=cfg.device)
    model.load_state_dict(state["model"])
    model.eval()
    return model


@torch.no_grad()
def ncsn_generate_and_denoise(
    checkpoint: Path,
    output_dir: Path,
    cfg: NCSNConfig,
    conditional: bool = False,
    noise_levels: Sequence[float] = (0.2, 0.4, 0.6),
) -> None:
    ensure_dir(output_dir)
    data_cfg = DataConfig(batch_size=16)
    train_loader, _ = mnist_dataloaders(data_cfg, normalize_to_minus1_1=True)
    model = load_ncsn_model(checkpoint, cfg, conditional)

    y_samples: Optional[torch.Tensor] = None
    if conditional:
        y_samples = torch.arange(0, 16, device=cfg.device) % cfg.num_classes
    samples = sample(model, cfg, num_samples=16, y=y_samples)
    save_grid((samples + 1) / 2.0, output_dir / "ncsn_samples.png", nrow=4)

    x_real, labels = next(iter(train_loader))
    x_real = x_real.to(cfg.device)[:16] * 2 - 1
    y = labels.to(cfg.device)[:16] if conditional else None

    for nl in noise_levels:
        noisy = x_real + nl * torch.randn_like(x_real)
        sigmas = torch.tensor([nl], device=cfg.device)
        denoised = annealed_langevin_dynamics(model, cfg, sigmas, noisy.clone(), y)
        save_grid((noisy + 1) / 2.0, output_dir / f"noisy_{nl:.2f}.png", nrow=4)
        save_grid((denoised + 1) / 2.0, output_dir / f"denoised_{nl:.2f}.png", nrow=4)


# Example: ncsn_generate_and_denoise(paths.images / "ncsn" / "ncsn.pt", paths.images / "ncsn_infer", NCSNConfig(), conditional=False)

## Quick demo run (short, inline)

- Runs 1 epoch EBM and NCSN (unconditional) with default configs.
- Uses small epochs to keep runtime manageable; for full quality use the scripts.
- Displays sample grids inline (not saved); ensure `torch` is installed and GPU is recommended.


# Analysis: EBM Results

- The EBM model is trained using contrastive divergence and Langevin sampling.
- Generated samples and denoised images are saved after each epoch.
- The quality of generated images improves with training, but may be blurry or lack diversity if the model or sampling steps are insufficient.
- Denoising works well for moderate noise, but fails for very high noise levels.
- See saved images in the results directory for qualitative evaluation.

In [ ]:
# Heavy generation cell: run full pipelines to produce report images (compute-heavy)
# Set RUN_FULL=True to execute; this cell includes OOM handling and CPU fallback.
RUN_FULL = False  # set to True to run full generation (use GPU runtime if available)
FORCE = False     # set True to retrain even if checkpoints exist
CPU_FALLBACK = True  # if GPU OOM occurs, retry on CPU with reduced settings

if RUN_FULL:
    print('Starting full report-generation run...')
    from pathlib import Path
    import torch
    from config import DataConfig, EBMConfig, NCSNConfig, RunPaths
    from ebm_train import train as full_train_ebm
    from ncsn_train import train as full_train_ncsn
    from ebm_infer import generate_and_denoise as ebm_generate_and_denoise, sample_and_save_trajectory as ebm_sample_traj
    from ncsn_infer import ncsn_generate_and_denoise, load_ncsn_model
    from ncsn_sampling import sample as ncsn_sample
    from utils import save_grid

    paths = RunPaths()
    paths.ensure()

    def try_train_ebm(cfg, out_dir):
        try:
            full_train_ebm(DataConfig(), cfg, out_dir)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and CPU_FALLBACK:
                print('EBM OOM on GPU; retrying on CPU with reduced steps...')
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                cfg.device = torch.device('cpu')
                cfg.langevin_steps = max(8, cfg.langevin_steps // 4)
                full_train_ebm(DataConfig(), cfg, out_dir)
            else:
                raise

    def try_train_ncsn(cfg, out_dir, conditional=False):
        try:
            full_train_ncsn(cfg, out_dir, conditional=conditional)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and CPU_FALLBACK:
                print('NCSN OOM on GPU; retrying on CPU with reduced config...')
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                cfg.device = torch.device('cpu')
                cfg.num_levels = max(3, cfg.num_levels // 2)
                cfg.langevin_steps = max(10, cfg.langevin_steps // 5)
                cfg.batch_size = min(cfg.batch_size, 32)
                full_train_ncsn(cfg, out_dir, conditional=conditional)
            else:
                raise

    # EBM full run
    ebm_out = paths.images / 'ebm'
    ebm_cfg = EBMConfig()
    ebm_cfg.epochs = 10
    ebm_cfg.sample_grid = 16
    ebm_cfg.langevin_steps = 60
    if FORCE or not (ebm_out / 'ebm_ckpt.pt').exists():
        try_train_ebm(ebm_cfg, ebm_out)
    else:
        print('EBM checkpoint exists; skipping training')

    # EBM inference
    try:
        ck = ebm_out / 'ebm_ckpt.pt'
        if ck.exists():
            ebm_generate_and_denoise(ck, paths.images / 'ebm_infer', DataConfig(), ebm_cfg)
            ebm_sample_traj(ck, paths.images / 'ebm_traj_demo', ebm_cfg, record_every=5)
        else:
            print('EBM checkpoint missing; inference skipped')
    except Exception as e:
        print('EBM inference failed:', e)

    # NCSN full run (unconditional + conditional)
    ncsn_out = paths.images / 'ncsn'
    ncsn_cond_out = paths.images / 'ncsn_cond'
    ncsn_cfg = NCSNConfig()
    ncsn_cfg.epochs = 30
    ncsn_cfg.num_levels = 10
    ncsn_cfg.langevin_steps = 150

    if FORCE or not (ncsn_out / 'ncsn.pt').exists():
        try_train_ncsn(ncsn_cfg, ncsn_out, conditional=False)
    else:
        print('NCSN checkpoint exists; skipping unconditional training')

    if FORCE or not (ncsn_cond_out / 'ncsn_cond.pt').exists():
        try_train_ncsn(ncsn_cfg, ncsn_cond_out, conditional=True)
    else:
        print('NCSN checkpoint exists; skipping conditional training')

    # NCSN inference
    try:
        ck1 = ncsn_out / 'ncsn.pt'
        if ck1.exists():
            ncsn_generate_and_denoise(ck1, paths.images / 'ncsn_infer', ncsn_cfg, conditional=False)
        else:
            print('NCSN unconditional checkpoint missing; inference skipped')
    except Exception as e:
        print('NCSN inference failed:', e)

    try:
        ck2 = ncsn_cond_out / 'ncsn_cond.pt'
        if ck2.exists():
            ncsn_generate_and_denoise(ck2, paths.images / 'ncsn_cond_infer', ncsn_cfg, conditional=True)
        else:
            print('NCSN conditional checkpoint missing; inference skipped')
    except Exception as e:
        print('NCSN conditional inference failed:', e)

    # NCSN trajectory generation
    try:
        if (ncsn_out / 'ncsn.pt').exists():
            model_n = load_ncsn_model(ncsn_out / 'ncsn.pt', ncsn_cfg, conditional=False)
            traj = ncsn_sample(model_n, ncsn_cfg, num_samples=16, return_trajectory=True, record_every=10)
            out_dir_n = paths.images / 'ncsn_traj_demo'
            out_dir_n.mkdir(parents=True, exist_ok=True)
            for i, frame in enumerate(traj):
                save_grid((frame + 1) / 2.0, out_dir_n / f'ncsn_traj_{i:03d}.png', nrow=4)
        else:
            print('Skipping NCSN trajectory generation; checkpoint missing')
    except Exception as e:
        print('NCSN trajectory generation failed:', e)

    print('Full report image generation complete. Look under the images/ directory.')
else:
    print('Heavy generation is disabled. Set RUN_FULL = True to enable.')

## Visualize demo losses

Plot the loss histories from the short demo runs to quickly inspect optimization behavior.


# Analysis: NCSN Results

- The NCSN model (unconditional and conditional) is trained with weighted DSM loss and annealed Langevin dynamics.
- Generated samples are sharper and more diverse than EBM, especially with sufficient noise levels and steps.
- Conditional NCSN can generate specific digits by conditioning on class labels.
- Denoising is effective for moderate noise, but extreme noise still degrades results.
- See saved images and GIFs for qualitative evaluation.

In [ ]:
import matplotlib.pyplot as plt

if "ebm_hist" in locals() and ebm_hist:
    plt.figure(figsize=(6, 4))
    plt.plot(ebm_hist["loss"])
    plt.title("EBM Loss (demo)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.savefig(paths.images / "ebm_demo_loss.png")
    plt.show()
else:
    print("Run the demo cell to populate ebm_hist.")

if "ncsn_hist" in locals() and ncsn_hist:
    plt.figure(figsize=(6, 4))
    plt.plot(ncsn_hist["loss"])
    plt.title("NCSN DSM Loss (demo)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.savefig(paths.images / "ncsn_demo_loss.png")
    plt.show()
else:
    print("Run the demo cell to populate ncsn_hist.")

In [ ]:
# Create GIFs from trajectory frames if available and display saved demo outputs
import imageio
from pathlib import Path
from IPython.display import Image, display


def make_gif_from_dir(frames_dir: Path, out_path: Path, fps: int = 6):
    frames = sorted(frames_dir.glob("*.png"))
    if not frames:
        print(f"No frames found in {frames_dir}")
        return None
    imgs = [imageio.imread(str(p)) for p in frames]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(str(out_path), imgs, fps=fps)
    print(f"Saved GIF: {out_path}")
    return out_path

# EBM demo outputs
ebm_demo = paths.images / "ebm_demo"
if ebm_demo.exists():
    print("EBM demo outputs:")
    for p in sorted(ebm_demo.glob("*.png")):
        display(Image(str(p)))
else:
    print("No EBM demo outputs found at:", ebm_demo)

# Create/show EBM trajectory GIF if possible
ebm_traj_dir = paths.images / "ebm_traj_demo"
if ebm_traj_dir.exists():
    gif = make_gif_from_dir(ebm_traj_dir, ebm_traj_dir / "ebm_traj.gif")
    if gif:
        display(Image(str(gif)))

# NCSN demo outputs
ncsn_demo = paths.images / "ncsn_demo"
if ncsn_demo.exists():
    print("NCSN demo outputs:")
    for p in sorted(ncsn_demo.glob("*.png")):
        display(Image(str(p)))
else:
    print("No NCSN demo outputs found at:", ncsn_demo)

# Create/show NCSN trajectory GIF if possible
ncsn_traj_dir = paths.images / "ncsn_traj_demo"
if ncsn_traj_dir.exists():
    gif = make_gif_from_dir(ncsn_traj_dir, ncsn_traj_dir / "ncsn_traj.gif")
    if gif:
        display(Image(str(gif)))


In [ ]:
## Saved demo outputs and GIFs

This cell collects saved images from the demo runs (`ebm_demo`, `ncsn_demo`) and any trajectory frames, creates GIFs when possible, and displays all results inline for easy inspection.


## Display saved figures if available

Checks for images produced by the script entrypoints (e.g., `images/ebm/ebm_samples_epoch10.png`) and shows them inline when present.


In [ ]:
from matplotlib import image as mpimg

candidates = [
    paths.images / "ebm" / "ebm_samples_epoch10.png",
    paths.images / "ebm" / "ebm_denoised_epoch10.png",
    paths.images / "ncsn" / "samples_epoch30.png",
    paths.images / "ncsn_cond" / "samples_epoch30.png",
    paths.images / "ncsn_infer" / "denoised_0.40.png",
]

for img_path in candidates:
    if img_path.exists():
        img = mpimg.imread(img_path)
        plt.figure(figsize=(5, 5))
        plt.axis("off")
        plt.title(img_path.name)
        plt.imshow(img)
        plt.show()
    else:
        print(f"Missing: {img_path}")